In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from catboost import CatBoostClassifier


columns = [
    'id', 'clump_thickness', 'unif_cell_size', 'unif_cell_shape', 
    'marh_adhesion', 'single_epith_cell_size', 'bare_nuclei', 
    'bland_chrom', 'norm_nucleoli', 'mitoses', 'class'
]
df = pd.read_csv(r'C:\Users\choua\Downloads\breast+cancer+wisconsin+original\breast-cancer-wisconsin.data', names=columns)


df.replace('?', np.nan, inplace=True)
df.drop(['id'], axis=1, inplace=True)


for col in df.columns:
    if col != 'class':
        df[col] = pd.to_numeric(df[col], errors='coerce')
df['class'] = pd.to_numeric(df['class'], errors='coerce')

df.fillna(df.median(numeric_only=True), inplace=True)
df = df.astype(int)  


X = df.drop(['class'], axis=1).values
y = df['class'].values 


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    verbose=0,
    random_seed=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Test Accuracy:", (y_pred == y_test).mean())
print("Classification Report:\n", classification_report(y_test, y_pred))

def predict_patient(patient_data):
    
    patient_array = np.array(patient_data).reshape(1, -1)
    prediction = model.predict(patient_array)[0]
    probabilities = model.predict_proba(patient_array)[0]
    
   
    class_idx = np.where(model.classes_ == prediction)[0][0]
    confidence = probabilities[class_idx]
    
    if confidence < 0.6:  
        return "Uncertain (Further Testing Needed)", confidence
    elif prediction == 2:
        return "Benign (Not Sick)", confidence
    elif prediction == 4:
        return "Malignant (Sick)", confidence
    else:
        return "Unknown Class", 0


new_patient_1 = [5, 1, 1, 1, 2, 1, 3, 1, 1]  
diagnosis_1, confidence_1 = predict_patient(new_patient_1)
print(f"\nDiagnosis: {diagnosis_1}, Confidence: {confidence_1:.2%}")

new_patient_2 = [10, 10, 10, 10, 10, 10, 10, 10, 10] 
diagnosis_2, confidence_2 = predict_patient(new_patient_2)
print(f"Diagnosis: {diagnosis_2}, Confidence: {confidence_2:.2%}")

Test Accuracy: 0.9642857142857143
Classification Report:
               precision    recall  f1-score   support

           2       0.97      0.98      0.97        90
           4       0.96      0.94      0.95        50

    accuracy                           0.96       140
   macro avg       0.96      0.96      0.96       140
weighted avg       0.96      0.96      0.96       140


Diagnosis: Benign (Not Sick), Confidence: 100.00%
Diagnosis: Malignant (Sick), Confidence: 99.99%


In [6]:
pip install catboost

  Obtaining dependency information for catboost from https://files.pythonhosted.org/packages/bd/9e/feae59f6226f742fa3fa30ae126e0941f443d460e7c0fa9f79cdf3ee488f/catboost-1.2.8-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for graphviz from https://files.pythonhosted.org/packages/00/be/d59db2d1d52697c6adc9eacaf50e8965b6345cc143f671e1ed068818d5cf/graphviz-0.20.3-py3-none-any.whl.metadata
  Obtaining dependency information for plotly from https://files.pythonhosted.org/packages/bf/6f/759d5da0517547a5d38aabf05d04d9f8adf83391d2c7fc33f904417d3ba2/plotly-6.1.2-py3-none-any.whl.metadata
  Obtaining dependency information for narwhals>=1.15.1 from https://files.pythonhosted.org/packages/c9/e0/ade8619846645461c012498f02b93a659e50f07d9d9a6ffefdf5ea2c02a0/narwhals-1.41.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/102.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/102.4 MB 682.7 kB/s eta 0:02:31
   ---------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
